In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ========================
# 1. LOAD THE DATA
# ========================
print("Loading shot_logs.csv...")
df = pd.read_csv('../data/raw/shot_logs.csv')

print(f"Total shots in dataset: {len(df):,}")
print(f"Shape: {df.shape}")
print("\nColumns:")
print(df.columns.tolist())

# Show sample
print("\nFirst 5 rows:")
print(df.head())

# ========================
# 2. BASIC INFORMATION
# ========================
print("\n" + "="*50)
print("DATASET INFORMATION")
print("="*50)
print(df.info())

print("\nMissing Values:")
missing = df.isnull().sum()
print(missing[missing > 0])

# ========================
# 3. DATA CLEANING
# ========================

# Create a clean copy
clean_df = df.copy()

# Rename columns for easier use
clean_df = clean_df.rename(columns={
    'SHOT_MADE_FLAG': 'made',
    'LOC_X': 'loc_x',
    'LOC_Y': 'loc_y',
    'CLOSE_DEF_DIST': 'defender_dist',
    'SHOT_DISTANCE': 'shot_distance',
    'PERIOD': 'period',
    'SHOT_CLOCK': 'shot_clock',
    'DRIBBLES': 'dribbles',
    'TOUCH_TIME': 'touch_time'
})

# Convert made to integer
clean_df['made'] = clean_df['made'].astype(int)

# ========================
# 4. FEATURE ENGINEERING
# ========================

# Distance to basket (very important feature)
clean_df['distance_to_basket'] = np.sqrt(clean_df['loc_x']**2 + clean_df['loc_y']**2)

# Shot angle (in degrees)
clean_df['shot_angle'] = np.degrees(np.arctan2(clean_df['loc_x'], clean_df['loc_y']))

# Shot zones
def get_shot_zone(row):
    if row['shot_distance'] <= 8:
        return 'Paint'
    elif row['shot_distance'] <= 22:
        return 'Mid-Range'
    else:
        return 'Three_Point'

clean_df['shot_zone'] = clean_df.apply(get_shot_zone, axis=1)

# Defender pressure category
def defender_pressure(dist):
    if dist <= 2: return 'Very Tight'
    elif dist <= 4: return 'Tight'
    elif dist <= 6: return 'Open'
    else: return 'Very Open'

clean_df['defender_pressure'] = clean_df['defender_dist'].apply(defender_pressure)

# Shot value (2 or 3 points)
clean_df['shot_value'] = np.where(clean_df['shot_distance'] > 22, 3, 2)

# ========================
# 5. FINAL CHECK
# ========================
print("\n" + "="*50)
print("FINAL CLEANED DATASET")
print("="*50)
print(clean_df.head())
print(f"\nFinal Shape: {clean_df.shape}")

# Save the cleaned version
clean_df.to_csv('../data/processed/cleaned_shots.csv', index=False)
print("\n✅ Cleaned data saved to: ../data/processed/cleaned_shots.csv")